In [0]:
"""
===============================================================
Module: Superstore Gold dimensional ETL pipeline orchestrator

Purpose:
    This module serves as the master orchestrator for processing Slowly Changing Dimensions (SCD2) 
    in the Gold layer of the Superstore Medallion Architecture. It handles reading from the Silver layer, 
    applying SCD2 logic for dimension tables, merging updates, managing soft deletes, collecting metrics, 
    and optimizing tables for query performance.

Key Features:
1. SCD2 Processing for Gold Dimensions:
    - Computes dimension timelines, including handling effective_from and effective_to dates.
    - Applies SCD2 logic to merge new and updated records, keeping track of historical data.
    - Handles soft deletes for outdated records, ensuring the current records remain intact.

2. Centralized Configuration for Multiple Dimensions:
    - Supports processing of multiple dimensions with configurations defined in a central `dim_config`.
    - Allows for different configuration settings for each dimension table (e.g., customer, product, etc.).

3. Incremental Data Handling:
    - Reads incremental data from the Silver layer and processes it efficiently for large datasets.
    - Tracks changes by comparing current data with previous states, avoiding unnecessary reprocessing.

4. Table layout and maintenance:
    - Each Gold dimension table is partitioned by the first column of `z_order_cols` in its config.
    - OPTIMIZE and VACUUM are left to Unity Catalog Predictive Optimization; this pipeline runs neither.

5. Error Handling & Logging:
    - Detailed logging using structured logging (e.g., master_run_id, layer_run_id) for observability.
    - In case of failures, proper error handling ensures that issues are logged and reported with stack trace information.

6. Metrics & Observability:
    - Tracks key metrics for every processed dimension table: rows read, rows inserted, deduplicated rows, etc.
    - Provides detailed performance insights via metrics collection, ensuring transparency in ETL processing.

7. Production-Ready:
    - Suitable for production-scale ETL processing, running on Databricks or any Spark-enabled platform.
    - Designed to handle large volumes of data and ensure scalability.

Best Practices:
- Ensure that `superstore_gold_dimension_config.yaml` is properly configured to map Silver tables to Gold tables.
- Review metrics regularly to track the progress and performance of the dimension tables in the Gold layer.
- Check the Gold tables for the proper handling of SCD2 changes and ensure soft deletes are applied correctly.

===============================================================
"""
# -------------------------------
# Core Python + Utility Imports
# -------------------------------
# Standard libraries for OS operations, timing, UUIDs, YAML config parsing, and module reloads
from pyspark.sql import SparkSession

import os, sys, time, yaml, importlib, uuid

# Used for handling Spark-specific exceptions
from pyspark.sql.utils import AnalysisException

# PySpark column functions for DataFrame transformations
from pyspark.sql.functions import col


# -------------------------------
# Spark Session Initialization
# -------------------------------
# Initialize or reuse existing Spark session
spark = SparkSession.builder.getOrCreate()

# Ensure consistent timestamp handling across pipeline (UTC standardization)
spark.conf.set("spark.sql.session.timeZone", "UTC")


# -------------------------------
# Runtime Environment Configuration
# -------------------------------
# Fetch runtime environment details from Databricks job context (task values)
env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

# Set environment variables for downstream modules (logging, config, audit, observability)
os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version


# -------------------------------
# Add Shared Code Paths
# -------------------------------
# this notebook lives at <bundle_root>/superstore_orchestrator/layer_orchestrator/<name>
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
# Path for shared utilities (logging, config management, platform helpers)
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")
# Path for Gold layer domain-specific framework (SCD2 logic, merges, metrics, etc.)
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_gold/core")   

# -------------------------------
# Core Framework Imports
# -------------------------------
# Logging and event tracking utilities
from superstore_logger import get_superstore_logger, log_event

# Execution context utility to retrieve master and layer run IDs
from superstore_orchestrator_execution_context import get_execution_context

# Platform constants (ensures consistent layer naming across pipelines)
from superstore_platform_constants import GOLD_LAYER

# Platform config utilities (schema + table helpers)
from superstore_platform_config import (
    get_silver_schema,
    get_gold_schema,
    get_metrics_schema,
    table
)

# -------------------------------
# Execution Context Initialization
# -------------------------------
# Retrieve pipeline execution identifiers for traceability and audit
execution_context = get_execution_context(dbutils, allow_standalone=True)
master_run_id = execution_context["master_run_id"]
layer_run_id = execution_context["layer_run_id"]


# -------------------------------
# Load Gold Dimension Configuration
# -------------------------------
# Read YAML configuration defining dimensions, keys, and processing rules
config_path = f"{BUNDLE_ROOT}/configs/superstore_gold_config/superstore_gold_dimension_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
    
# Extract dimension configuration block
dim_config = config["dim_config"]


# -------------------------------
# Import Gold Dimension Framework
# -------------------------------
# Load reusable SCD2 + Gold processing framework
# Reload ensures latest code changes are applied during notebook execution (dev mode)
import superstore_gold_dimension_framework
importlib.reload(superstore_gold_dimension_framework)

from superstore_gold_dimension_framework import (
    read_silver_table,              # reads Silver layer source data
    prepare_scd2_columns,           # prepares hashing + SCD2 metadata columns
    compute_scd2_timeline,          # builds SCD2 effective dating logic
    create_gold_table_if_not_exists,# ensures Gold table existence
    merge_into_gold_table_scd2,     # performs SCD2 MERGE into Gold
    handle_soft_deletes,            # applies soft delete logic
    collect_metrics                 # writes Gold layer metrics
)
from superstore_backfill_utils import (
    get_backfill_config,
    validate_backfill_impact
)

# -------------------------------
# Logger Initialization
# -------------------------------
# Create orchestrator-specific logger for Gold dimensional pipeline
logger_dim_orchestrator = get_superstore_logger("superstore_gold_dimensional_orchestrator")

# Log pipeline start event (audit + observability)
log_event(
    logger_dim_orchestrator,
    "INFO",
    "Initiating Superstore Gold layer ETL dimensional orchestrator",
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=GOLD_LAYER
)

# Get backfill_config
backfill_config = get_backfill_config(dbutils, allow_full_refresh=False)

log_event(
    logger_dim_orchestrator,
    "INFO",
    "Gold dimensional run configuration loaded",
    run_mode=backfill_config["mode"],
    dry_run=backfill_config["dry_run"],
    is_windowed=backfill_config["is_windowed"],
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=GOLD_LAYER
)

# -------------------------------
# Dry run
# -------------------------------
# Report what this run would touch, then exit before any write. Every task
# that writes must honour dry_run: a preview that writes to some layers while
# the DAG reports success is worse than having no flag, because it creates
# false confidence exactly when an operator is being careful.
if backfill_config["dry_run"]:
    for _cfg_name, cfg in dim_config.items():
        source_tbl = table(get_silver_schema(), cfg["silver_tbl"])
        impact = validate_backfill_impact(
            spark,
            source_tbl,
            backfill_config,
            date_column="ingestion_date"
        )
        log_event(
            logger_dim_orchestrator,
            "INFO",
            f"Dry-run impact for {source_tbl}",
            run_mode=backfill_config["mode"],
            impact=impact,
            master_run_id=master_run_id,
            layer_run_id=layer_run_id,
            layer=GOLD_LAYER
        )
    dbutils.notebook.exit(
        "DRY_RUN_COMPLETED run_mode=" + backfill_config["mode"] + " layer=gold_dimensions (no data written)"
    )


# -------------------------------
# Gold Dimensional ETL Orchestrator
# -------------------------------
def run_all_dim_pipelines(spark, master_run_id: str):
    log_event(
        logger_dim_orchestrator,
        "INFO",
        "Starting Gold dimensional pipelines for all configured tables",
        total_tables=len(dim_config),
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=GOLD_LAYER
    )

    for table_name, cfg in dim_config.items():

        # -----------------------------
        # Dynamically resolve full catalog.schema.table
        # -----------------------------
        silver_tbl = table(get_silver_schema(), cfg["silver_tbl"])
        gold_tbl = table(get_gold_schema(), cfg["gold_tbl"])
        metrics_table = table(get_metrics_schema(), cfg["metrics_table"])
    
        layer_name = cfg.get("layer_name")
        table_type = cfg.get("table_type")
        dim_type= cfg.get("dim_type")
        required_columns = cfg["required_columns"]
        natural_keys = cfg["natural_keys"]
        column_mapping = cfg["column_mapping"]
        hash_column = cfg["hash_column"]
        z_order_cols = cfg.get("z_order_cols", [])
        entity_id_col = natural_keys[0]  # Assumes single natural key for SCD2

        start_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
        start_time = time.time()
        has_incremental_rows = False

        try:
            log_event(
                logger_dim_orchestrator,
                "INFO",
                f"Starting dimensional pipeline for {gold_tbl}",
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 1: Read Silver table
            df_silver = read_silver_table(
                spark=spark,
                silver_table=silver_tbl,
                gold_table=gold_tbl,
                backfill_config=backfill_config,  # backfill
                required_columns=required_columns,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # -----------------------------------------
            # Capture GOLD BEFORE ANY DECISION
            # -----------------------------------------
            try:
                dim_gold_before_merge = spark.table(gold_tbl).filter(col("is_current") == True)
            except AnalysisException:
                dim_gold_before_merge = None
            
            # -----------------------------------------
            # CASE A: Silver table does NOT exist
            # -----------------------------------------
            if df_silver is None:
                log_event(
                    logger_dim_orchestrator,
                    "INFO",
                    f"No {silver_tbl} table found in Silver for {gold_tbl} ",
                    table=gold_tbl,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER    
                )
                collect_metrics(
                    spark,
                    repaired_df=None,
                    merge_metrics=None,
                    is_initial_load=None,
                    gold_tbl=gold_tbl,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    table_name=gold_tbl,
                    table_type=table_type,
                    entity_id_column=entity_id_col,
                    metrics_table=metrics_table,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    start_ts=start_ts,
                    end_ts=start_ts,
                    duration_secs=0,
                    backfill_config=backfill_config
                )
                continue


            # -----------------------------------------
            # CASE B: Silver exists but EMPTY
            # -----------------------------------------
            has_incremental_rows = df_silver.limit(1).count() > 0
            if not has_incremental_rows:
                log_event(
                    logger_dim_orchestrator,
                    "INFO",
                    f"No new rows found in  {silver_tbl} for processing; skipping",
                    table=gold_tbl,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )
                collect_metrics(
                    spark,
                    repaired_df=df_silver,
                    merge_metrics=None,
                    is_initial_load=None,
                    gold_tbl=gold_tbl,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    table_name=gold_tbl,
                    table_type=table_type,
                    entity_id_column=entity_id_col,
                    metrics_table=metrics_table,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    start_ts=start_ts,
                    end_ts=start_ts,
                    duration_secs=0,
                    backfill_config=backfill_config
                )
            
            if has_incremental_rows:
                # Step 2: Prepare SCD2 columns
                df_scd2 = prepare_scd2_columns(
                    silver_df=df_silver,
                    dim_type=dim_type,
                    entity_columns=required_columns,
                    hash_column=hash_column,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    # Keys are excluded from Unknown substitution: they are
                    # the fatal tier, so they can never be null here.
                    natural_keys=natural_keys
                )

                # Step 3: Compute timeline
                df_timeline = compute_scd2_timeline(
                    scd_df=df_scd2,
                    entity_id_column=entity_id_col,
                    hash_column=hash_column,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                # Step 4: Ensure Gold table exists
                create_gold_table_if_not_exists(
                    spark=spark,
                    repaired_df=df_timeline,
                    gold_tbl=gold_tbl,
                    partition_column=z_order_cols[0] if z_order_cols else None,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                # ---------------------------------------------------------
                # Detect INITIAL LOAD condition for dimensions Gold table 
                # ---------------------------------------------------------
                # is_initial_load → TRUE when the table exists but contains no data
                #
                # In this pipeline:
                # - Gold table is typically created earlier (even if empty)
                # - So initial load = first time data is being written into an empty table
                #
                # This flag is later used to:
                # - Set execution_mode = "INITIAL_LOAD"
                # - Differentiate first-time load vs incremental SCD2 processing
                # ---------------------------------------------------------
                is_initial_load = spark.table(gold_tbl).limit(1).count() == 0
       

                # Step 5: Merge timeline into Gold (SCD2)
                results=merge_into_gold_table_scd2(
                    spark=spark,
                    repaired_df=df_timeline,
                    gold_tbl=gold_tbl,
                    entity_id_column=entity_id_col,
                    hash_column=hash_column,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                # Extract values safely
                merge_metrics = results.get("merge_metrics", {})

                # Step 6: Handle soft deletes
                soft_deleted_rows=handle_soft_deletes(
                    spark=spark,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    silver_df=df_silver,
                    gold_tbl=gold_tbl,
                    entity_id_column=entity_id_col,
                    silver_table=silver_tbl   
                )

                end_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
                duration_secs = int(time.time() - start_time)

                collect_metrics(
                    spark,
                    repaired_df=df_silver,
                    merge_metrics=merge_metrics,
                    is_initial_load=is_initial_load,
                    gold_tbl=gold_tbl,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    table_name=gold_tbl,
                    table_type=table_type,
                    entity_id_column=entity_id_col,
                    metrics_table=metrics_table,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    start_ts=start_ts,
                    end_ts=end_ts,
                    duration_secs=duration_secs,
                    backfill_config=backfill_config
                )

                log_event(
                    logger_dim_orchestrator,
                    "INFO",
                    f"Completed dimensional pipeline for table: {gold_tbl}",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

        except Exception as e:
            run_status = "FAILED"
            notes = str(e)
            log_event(
                logger_dim_orchestrator,
                "ERROR",
                f"Dimensional pipeline failed for table {gold_tbl}: {e}",
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )
            raise

    log_event(
        logger_dim_orchestrator,
        "INFO",
        "Completed all Gold dimensional pipelines",
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=GOLD_LAYER
    )

# -------------------------------
# Run ETL
# -------------------------------
if __name__ == "__main__":
    run_all_dim_pipelines(spark, master_run_id)
